In [0]:
%pip install pendulum

In [0]:
import requests
import json
import pendulum
from pathlib import Path
import pandas as pd


In [0]:
yesterday = pendulum.now("America/Bogota").subtract(days=1)
fecha_ini= yesterday.to_date_string()
print(yesterday)
print(fecha_ini)

In [0]:
# Detecta el catálogo actual (suele ser 'workspace' o 'main')
current_catalog = spark.sql("select current_catalog()").first()[0]
catalog = current_catalog
schema = "raw"
volume = "uses"

# Crea esquema y volumen si no existen
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{schema}.{volume}")

In [0]:
# Elimina la carpeta del dia en que se este ejecutando el proceso
path_delete = f"/Volumes/workspace/raw/uses/{yesterday.year:04d}/{yesterday.month:02d}/{yesterday.day:02d}"

# Eliminar solo esa carpeta
dbutils.fs.rm(path_delete, recurse=True)

#print(f"Eliminado: {path_delete}")


In [0]:
query = f"select max(a.id) from workspace.silver.uses a;"
max_id = spark.sql(query).first()[0]
print(max_id)



In [0]:
# Crear widgets

dbutils.widgets.text("output_dir", "/Volumes/workspace/raw/uses", "Output Directory")
dbutils.widgets.text("trx_id","99999999999","Transaction ID")
dbutils.widgets.text("enpoint", "https://api.trafpay.cl/api-mifare/transactions/GetTransacionByID?ProjectID=1&TransactionID={}", "API endpoint")
dbutils.widgets.text("timeout", "300", "Time Out")

# Leer valores de los widgets

start_date = yesterday.date()
output_dir = Path(dbutils.widgets.get("output_dir"))
trx_id = dbutils.widgets.get("trx_id")
endpoint_dir = dbutils.widgets.get("enpoint")
API_URL = dbutils.widgets.get("enpoint").format(max_id) #TRX_ID Automatico desde tabla
timeout = int(dbutils.widgets.get("timeout"))

In [0]:
print(start_date)
print(API_URL)


In [0]:
saved_files = []
session = requests.Session()

#Crear el directorio donde se guardan los archivo json

current_date = yesterday #Fecha Automatica

daily_output_dir = (
    output_dir
    / f"{current_date.year:04d}"
    / f"{current_date.month:02d}"
    / f"{current_date.day:02d}"
)

daily_output_dir.mkdir(parents=True, exist_ok=True)

file_path = daily_output_dir / "uses.json"

# Realizar la llamada a la API con su key
api_key = dbutils.secrets.get(scope="key-apis", key="usosapikey")
headers = {"apikey": api_key}
response = session.get(API_URL, headers=headers, timeout=timeout)

if response.status_code != 200:
    raise Exception(f"Error en API: {response.status_code} - {response.text}")

# Guardar directamente el JSON
file_path.write_bytes(response.content)

# Guardar ruta en lista
saved_files.append(str(file_path))

current_date = current_date.add(days=1)
print(f"Archivos guardados: {len(saved_files)}")

In [0]:
print(daily_output_dir)